In [11]:
from typing_extensions import TypedDict, Literal, Annotated
from typing import List
from langgraph.types import Send
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from pydantic import BaseModel
from operator import add

llm = init_chat_model("openai:gpt-4o")

In [12]:
class State(TypedDict):

    document: str
    final_summary: str
    summaries: Annotated[list[dict], add]

In [13]:
def summarize_p(args):
    paragraph = args["paragraph"]
    index = args["index"]
    response = llm.invoke(
        f"Write a 3-sentence summary for this paragraph: {paragraph}",
    )
    return {
        "summaries": [
            {
                "summary": response.content,
                "index": index,
            }
        ],
    }


def dispatch_summarizers(state: State):
    chunks = state["document"].split("\n\n")
    return [
        Send("summarize_p", {"paragraph": chunk, "index": index})
        for index, chunk in enumerate(chunks)
    ]


def final_summary(state: State):
    response = llm.invoke(
        f"Using the following summaries, give me a final one {state["summaries"]}"
    )
    return {
        "final_summary": response.content,
    }

In [14]:
graph_builder = StateGraph(State)

graph_builder.add_node("summarize_p", summarize_p)
graph_builder.add_node("final_summary", final_summary)


graph_builder.add_conditional_edges(
    START,
    dispatch_summarizers,
    ["summarize_p"],
)

graph_builder.add_edge("summarize_p", "final_summary")
graph_builder.add_edge("final_summary", END)


graph = graph_builder.compile()

In [15]:
with open("fed_transcript.md", "r", encoding="utf-8") as file:
    document = file.read()


for chunk in graph.stream(
    {"document": document},
    stream_mode="updates",
):
    print(chunk, "\n")

{'summarize_p': {'summaries': [{'summary': 'Inflation for goods has increased, reaching higher levels compared to earlier in the year. Meanwhile, disinflation persists in the services sector. Additionally, short-term inflation indicators have been affected by changes in tariffs, leading to an imbalanced situation.', 'index': 11}]}} 

{'summarize_p': {'summaries': [{'summary': 'Wage growth has been slowing down, yet it remains higher than the rate of inflation. This indicates that people are still experiencing increases in their real income despite the moderation in wage growth. As a result, individuals are potentially enjoying improved purchasing power.', 'index': 8}]}} 

{'summarize_p': {'summaries': [{'summary': 'In August, the unemployment rate increased slightly to 4.3 percent. Despite this rise, it has remained relatively stable over the past year. Overall, the unemployment rate is still considered low.', 'index': 5}]}} 

{'summarize_p': {'summaries': [{'summary': 'Inflation has d